# Data Cleaning

This notebook cleans all five raw files and writes the cleaned versions to
`processed/`. It does not repeat the exploration already done in the EDA notebooks; it
only applies the fixes those notebooks identified. Each section names the file it
cleans and lists the fixes applied, in the same order they were found.

Nothing here is dropped silently. Rows or values treated as missing are set to `NaN`
rather than filled with a guess, so any modelling step downstream can decide how to
handle them.

In [1]:
import pandas as pd
import numpy as np

RAW = '../raw/'
OUT = '../processed/'

import os
os.makedirs(OUT, exist_ok=True)

## 1. bank_profiles.csv

Fixes: drop exact duplicate rows, strip `$`/`,` from `total_assets_usd`, strip `%`
from `baseline_liquidity_ratio_pct`, convert missing tokens in `baseline_car_pct` to
`NaN`, set the one negative `total_assets_usd` value to `NaN`, and fix spelling in
`size_tier` and `sector_concentration`.

In [2]:
bank = pd.read_csv(RAW + 'bank_profiles.csv', dtype=str, keep_default_na=False)

bank = bank.drop_duplicates()

bank['total_assets_usd'] = pd.to_numeric(
    bank['total_assets_usd'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip(),
    errors='coerce')
bank.loc[bank['total_assets_usd'] < 0, 'total_assets_usd'] = np.nan

bank['total_loans_usd'] = pd.to_numeric(bank['total_loans_usd'], errors='coerce')
bank['deposit_base_usd'] = pd.to_numeric(bank['deposit_base_usd'], errors='coerce')

CAR_MISSING = ['', 'unknown', '-999', '-']
bank['baseline_car_pct'] = bank['baseline_car_pct'].str.strip().replace(CAR_MISSING, np.nan)
bank['baseline_car_pct'] = pd.to_numeric(bank['baseline_car_pct'], errors='coerce')

bank['baseline_liquidity_ratio_pct'] = pd.to_numeric(
    bank['baseline_liquidity_ratio_pct'].str.replace('%', '', regex=False).str.strip(),
    errors='coerce')

bank['baseline_roa_pct'] = pd.to_numeric(
    bank['baseline_roa_pct'].str.strip().replace(['unknown', ''], np.nan), errors='coerce')

bank['bank_risk_factor'] = pd.to_numeric(bank['bank_risk_factor'], errors='coerce')

SIZE_TIER_MAP = {'small': 'small', 'medium': 'medium', 'med': 'medium', 'medim': 'medium',
                  'large': 'large', 'larg': 'large'}
bank['size_tier'] = bank['size_tier'].str.strip().str.lower().map(SIZE_TIER_MAP)

SECTOR_CONC_MAP = {'diversified': 'diversified', 'concentrated': 'concentrated',
                    'concentratd': 'concentrated'}
bank['sector_concentration'] = bank['sector_concentration'].str.strip().str.lower().map(SECTOR_CONC_MAP)

sector_cols = [c for c in bank.columns if c.startswith('sector_wt_')]
for c in sector_cols:
    bank[c] = pd.to_numeric(bank[c], errors='coerce')

print('Cleaned shape:', bank.shape)
print('Missing values per column:')
print(bank.isna().sum()[bank.isna().sum() > 0])

Cleaned shape: (40, 20)
Missing values per column:
total_assets_usd    1
baseline_car_pct    6
baseline_roa_pct    4
dtype: int64


In [3]:
bank.to_csv(OUT + 'bank_profiles_clean.csv', index=False)
print('Saved processed/bank_profiles_clean.csv')

Saved processed/bank_profiles_clean.csv


## 2. loan_portfolio.csv

Fixes: drop exact duplicate rows, map over 40 sector spellings down to 10 sectors,
convert `pd_annual` percentage-formatted rows to the same decimal scale as the rest,
convert missing tokens in `pd_annual` and `lgd` to `NaN`, and strip thousand separators
from `ead`.

In [4]:
loan = pd.read_csv(RAW + 'loan_portfolio.csv', dtype=str, keep_default_na=False)

loan = loan.drop_duplicates()

SECTOR_MAP = {
    'technology': 'Technology', 'tech': 'Technology', 'technlogy': 'Technology',
    'techno logy': 'Technology',
    'healthcare': 'Healthcare', 'health care': 'Healthcare',
    'real estate': 'Real_Estate', 'real  estate': 'Real_Estate', 'real-estate': 'Real_Estate',
    'real_estate': 'Real_Estate',
    'energy': 'Energy', 'enrgy': 'Energy',
    'industrials': 'Industrials', 'industrial': 'Industrials', 'industrails': 'Industrials',
    'retail': 'Retail', 'retial': 'Retail',
    'telecom': 'Telecom', 'telcom': 'Telecom', 'tele com': 'Telecom', 'telecomm': 'Telecom',
    'consumer': 'Consumer', 'consumer goods': 'Consumer', 'consumr': 'Consumer',
    'utilities': 'Utilities', 'utility': 'Utilities', 'utilties': 'Utilities',
    'financials': 'Financials', 'financail': 'Financials', 'financails': 'Financials',
    'finance': 'Financials', 'financial': 'Financials',
}
loan['sector'] = loan['sector'].str.strip().str.lower().map(SECTOR_MAP)

pd_annual_raw = loan['pd_annual'].str.strip()
PD_MISSING = ['', 'n/a', 'na', '-', '--', 'none', 'missing', 'null', 'unknown']
is_missing = pd_annual_raw.str.lower().isin(PD_MISSING)
is_pct = pd_annual_raw.str.contains('%', na=False)

pd_annual_numeric = pd.to_numeric(pd_annual_raw.str.replace('%', '', regex=False), errors='coerce')
pd_annual_numeric = np.where(is_pct, pd_annual_numeric / 100, pd_annual_numeric)
pd_annual_numeric = pd.Series(pd_annual_numeric, index=loan.index)
pd_annual_numeric[is_missing] = np.nan
loan['pd_annual'] = pd_annual_numeric

LGD_MISSING = ['', 'na', 'n.a.', 'not available', '-']
loan['lgd'] = loan['lgd'].str.strip().str.lower().replace(LGD_MISSING, np.nan)
loan['lgd'] = pd.to_numeric(loan['lgd'], errors='coerce')

loan['ead'] = pd.to_numeric(loan['ead'].str.replace(',', '', regex=False), errors='coerce')
loan['rwa'] = pd.to_numeric(loan['rwa'], errors='coerce')
loan['loan_amount'] = pd.to_numeric(loan['loan_amount'], errors='coerce')

print('Cleaned shape:', loan.shape)
print('Sector value counts:')
print(loan['sector'].value_counts(dropna=False))
print()
print('Missing values per column:')
print(loan.isna().sum()[loan.isna().sum() > 0])

Cleaned shape: (3000, 7)
Sector value counts:
sector
Retail         449
Telecom        371
Energy         363
Technology     352
Consumer       346
Healthcare     303
Real_Estate    274
Industrials    269
Utilities      158
Financials     115
Name: count, dtype: int64

Missing values per column:
pd_annual    74
lgd          60
dtype: int64


In [5]:
loan.to_csv(OUT + 'loan_portfolio_clean.csv', index=False)
print('Saved processed/loan_portfolio_clean.csv')

Saved processed/loan_portfolio_clean.csv


## 3. macro_scenarios.csv

Fixes: drop exact duplicate rows, map severity spellings down to 5 categories
(unreadable labels become `NaN` rather than a guess), strip the trailing `bps` label
from `credit_spread_bps`, and convert missing tokens in `rate_shock_pp` to `NaN`.

In [6]:
macro = pd.read_csv(RAW + 'macro_scenarios.csv', dtype=str, keep_default_na=False)

macro = macro.drop_duplicates()

SEVERITY_MAP = {
    'baseline': 'baseline',
    'mild': 'mild', 'mld': 'mild',
    'moderate': 'moderate', 'moderat': 'moderate',
    'adverse': 'adverse', 'advers': 'adverse',
    'severe': 'severe', 'sever': 'severe',
}
macro['scenario_severity'] = macro['scenario_severity'].str.strip().str.lower().map(SEVERITY_MAP)

MACRO_MISSING = {'', 'na', 'n/a', 'n.a.', 'not available', '-', '--', 'missing',
                  'unknown', 'null', 'none', 'tbd'}

def parse_bps(v):
    v = v.strip().lower()
    if v in MACRO_MISSING:
        return np.nan
    v = v.replace('bps', '').strip()
    try:
        return float(v)
    except ValueError:
        return np.nan

def parse_numeric(v):
    v = v.strip().lower()
    if v in MACRO_MISSING:
        return np.nan
    try:
        return float(v)
    except ValueError:
        return np.nan

macro['credit_spread_bps'] = macro['credit_spread_bps'].apply(parse_bps)
macro['rate_shock_pp'] = macro['rate_shock_pp'].apply(parse_numeric)

for col in ['gdp_shock_pp', 'unemp_shock_pp', 'inflation_shock_pp', 'fx_devaluation_pct', 'stress_intensity']:
    macro[col] = pd.to_numeric(macro[col], errors='coerce')

print('Cleaned shape:', macro.shape)
print('Missing values per column:')
print(macro.isna().sum()[macro.isna().sum() > 0])

Cleaned shape: (500, 9)
Missing values per column:
scenario_severity    10
rate_shock_pp        20
dtype: int64


In [7]:
macro.to_csv(OUT + 'macro_scenarios_clean.csv', index=False)
print('Saved processed/macro_scenarios_clean.csv')

Saved processed/macro_scenarios_clean.csv


## 4. macro_stress_scenarios.csv

Fixes: normalise whitespace and underscores in `sector`, lower-case and strip
`scenario`, and convert the numeric columns to floats. The `gfc_like` / `covid_like`
gap noted in the EDA notebook is a data-coverage issue, not something a cleaning step
can fix, so it is left as-is and documented in the README instead.

In [8]:
stress = pd.read_csv(RAW + 'macro_stress_scenarios.csv', dtype=str, keep_default_na=False)

print('Duplicate rows:', stress.duplicated().sum())

stress['scenario'] = stress['scenario'].str.strip().str.lower()

SECTOR_MAP = {
    'technology': 'Technology', 'tech': 'Technology',
    'healthcare': 'Healthcare', 'health care': 'Healthcare',
    'real estate': 'Real_Estate',
    'energy': 'Energy',
    'industrials': 'Industrials', 'industrial': 'Industrials',
    'retail': 'Retail', 'retial': 'Retail',
    'telecom': 'Telecom', 'tele com': 'Telecom',
    'consumer': 'Consumer', 'consumer goods': 'Consumer',
    'utilities': 'Utilities', 'utility': 'Utilities',
    'financials': 'Financials', 'finance': 'Financials', 'financial': 'Financials',
}
stress['sector'] = stress['sector'].str.strip().str.lower().str.replace('_', ' ').map(SECTOR_MAP)

numeric_cols = ['gdp_shock_pp', 'unemp_shock_pp', 'rate_shock_pp', 'credit_spread_bps',
                 'inflation_shock_pp', 'fx_devaluation_pct', 'pd_multiplier',
                 'base_lgd', 'stressed_lgd']
for col in numeric_cols:
    stress[col] = pd.to_numeric(stress[col], errors='coerce')

print('Cleaned shape:', stress.shape)
print('Sector value counts:')
print(stress['sector'].value_counts(dropna=False))
print()
print('Missing values per column:')
print(stress.isna().sum()[stress.isna().sum() > 0])

Duplicate rows: 0
Cleaned shape: (60, 11)
Sector value counts:
sector
Technology     6
Healthcare     6
Real_Estate    6
Energy         6
Industrials    6
Retail         6
Telecom        6
Consumer       6
Utilities      6
Financials     6
Name: count, dtype: int64

Missing values per column:
pd_multiplier    5
dtype: int64


In [9]:
stress.to_csv(OUT + 'macro_stress_scenarios_clean.csv', index=False)
print('Saved processed/macro_stress_scenarios_clean.csv')

Saved processed/macro_stress_scenarios_clean.csv


## 5. bank_stress_simulated_panel.csv

Fixes: drop exact duplicate rows, map `bank_condition` and `scenario_severity`
spellings down to their real categories, convert missing tokens in `car_after_pct` to
`NaN`, convert the `-1` sentinel and missing tokens in `liquidity_after_pct` to `NaN`,
correct the apparent decimal-point-shift error in `roa_after_pct`, and fix spelling in
`size_tier` and `sector_concentration`.

In [10]:
panel = pd.read_csv(RAW + 'bank_stress_simulated_panel.csv', dtype=str, keep_default_na=False)

panel = panel.drop_duplicates()

CONDITION_MAP = {
    'healthy': 'Healthy', 'helathy': 'Healthy',
    'stressed': 'Stressed', 'stresed': 'Stressed',
    'critical': 'Critical', 'critcal': 'Critical',
}
panel['bank_condition'] = panel['bank_condition'].str.strip().str.lower().map(CONDITION_MAP)

SEVERITY_MAP = {
    'baseline': 'baseline',
    'mild': 'mild', 'mld': 'mild',
    'moderate': 'moderate', 'moderat': 'moderate',
    'adverse': 'adverse', 'advers': 'adverse',
    'severe': 'severe', 'sever': 'severe',
}
panel['scenario_severity'] = panel['scenario_severity'].str.strip().str.lower().map(SEVERITY_MAP)

SIZE_TIER_MAP = {'small': 'small', 'medium': 'medium', 'med': 'medium', 'medim': 'medium',
                  'large': 'large', 'larg': 'large'}
panel['size_tier'] = panel['size_tier'].str.strip().str.lower().map(SIZE_TIER_MAP)

SECTOR_CONC_MAP = {'diversified': 'diversified', 'concentrated': 'concentrated',
                    'concentratd': 'concentrated'}
panel['sector_concentration'] = panel['sector_concentration'].str.strip().str.lower().map(SECTOR_CONC_MAP)

CAR_MISSING = ['', 'missing', 'unknown', ' ']
panel['car_after_pct'] = panel['car_after_pct'].str.strip().replace(CAR_MISSING, np.nan)
panel['car_after_pct'] = pd.to_numeric(panel['car_after_pct'], errors='coerce')

LIQ_MISSING = ['', 'unknown', 'tbd']
panel['liquidity_after_pct'] = panel['liquidity_after_pct'].str.strip().str.lower().replace(LIQ_MISSING, np.nan)
panel['liquidity_after_pct'] = pd.to_numeric(panel['liquidity_after_pct'], errors='coerce')
panel.loc[panel['liquidity_after_pct'] == -1, 'liquidity_after_pct'] = np.nan

panel['roa_after_pct'] = pd.to_numeric(panel['roa_after_pct'], errors='coerce')
implausible = (panel['roa_after_pct'] < -10) | (panel['roa_after_pct'] > 10)
print('Rows corrected for the decimal-shift ROA error:', implausible.sum())
panel.loc[implausible, 'roa_after_pct'] = panel.loc[implausible, 'roa_after_pct'] / 100

for col in ['total_assets_usd', 'total_loans_usd', 'baseline_car_pct',
            'baseline_liquidity_ratio_pct', 'baseline_roa_pct', 'gdp_shock_pp',
            'unemp_shock_pp', 'rate_shock_pp', 'credit_spread_bps', 'inflation_shock_pp',
            'fx_devaluation_pct', 'weighted_pd_multiplier', 'projected_npl_ratio_pct',
            'stressed_el_rate_pct', 'incremental_credit_loss_usd', 'car_after_pct']:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors='coerce')

print('Cleaned shape:', panel.shape)
print('bank_condition value counts:')
print(panel['bank_condition'].value_counts(dropna=False))
print()
print('Missing values per column:')
print(panel.isna().sum()[panel.isna().sum() > 0])

Rows corrected for the decimal-shift ROA error: 15


Cleaned shape: (20000, 24)
bank_condition value counts:
bank_condition
Healthy     10291
Stressed     6557
Critical     3152
Name: count, dtype: int64

Missing values per column:
incremental_credit_loss_usd    120
car_after_pct                  161
liquidity_after_pct            150
dtype: int64


In [11]:
panel.to_csv(OUT + 'bank_stress_simulated_panel_clean.csv', index=False)
print('Saved processed/bank_stress_simulated_panel_clean.csv')

Saved processed/bank_stress_simulated_panel_clean.csv


## 6. Final check

A quick listing of everything written to `processed/`, with row counts, to confirm the
cleaning ran end to end.

In [12]:
import os
for f in sorted(os.listdir(OUT)):
    df = pd.read_csv(OUT + f)
    print(f'{f}: {df.shape[0]} rows, {df.shape[1]} columns')

bank_profiles_clean.csv: 40 rows, 20 columns
bank_stress_simulated_panel_clean.csv: 20000 rows, 24 columns
loan_portfolio_clean.csv: 3000 rows, 7 columns
macro_scenarios_clean.csv: 500 rows, 9 columns
macro_stress_scenarios_clean.csv: 60 rows, 11 columns
